# MédiRDV — Rapport d'analyse data science

**AO-CLB-2026-01 — Clinique Les Baobabs**

Ce notebook présente l'analyse complète du jeu de données synthétique MédiRDV :
1. Exploration et qualité des données
2. Effets réalistes attendus (pics de demande, saisonnalité)
3. Taux d'absence et facteurs associés
4. Prévision de la demande (EF-24)
5. Score de risque d'absence (EF-25)
6. Limites et perspectives

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(RACINE / "data" / "propre" / "rendez_vous_medirdv_propre.csv", encoding="utf-8")
df["date_rdv"] = pd.to_datetime(df["date_rdv"], errors="coerce")
df["date_prise"] = pd.to_datetime(df["date_prise"], errors="coerce")
df["date_naissance"] = pd.to_datetime(df["date_naissance"], errors="coerce")
df["jour_semaine"] = df["date_rdv"].dt.day_name()
df["mois"] = df["date_rdv"].dt.to_period("M").astype(str)
df["heure_int"] = df["heure_rdv"].str[:2].astype(int)
df["age"] = ((df["date_rdv"] - df["date_naissance"]).dt.days / 365.25).round(0)
df["delai"] = (df["date_rdv"] - df["date_prise"]).dt.days

print(f"Dimensions : {df.shape}")
print(f"Période : {df['date_rdv'].min().date()} → {df['date_rdv'].max().date()}")
print(f"Spécialités : {df['specialite'].nunique()}")
print(f"Médecins : {df['medecin_id'].nunique()}")
print(f"Patients : {df['patient_id'].nunique()}")

Dimensions : (2015, 25)
Période : 2025-10-01 → 2026-12-31
Spécialités : 9
Médecins : 30
Patients : 833


## 1. Exploration et qualité des données

Le fichier brut (2040 lignes) a été nettoyé via `scripts/nettoyage.py` :

- Suppression de 25 doublons d'`id_rdv`
- Normalisation des dates (format mixte `JJ/MM/AAAA` et `AAAA-MM-JJ`)
- Normalisation des spécialités (9 valeurs officielles)
- Normalisation du sexe (`F`/`M`), statut (5 valeurs), confirme (`Oui`/`Non`)
- Validation des téléphones (format sénégalais)
- Correction des durées invalides (15/20/30 uniquement)

Résultat : **2015 lignes propres**.

In [2]:
print("Répartition par statut :")
print(df["statut"].value_counts())
print(f"\nTop 5 spécialités :")
print(df["specialite"].value_counts().head())
print(f"\nCanaux de prise de RDV :")
print(df["canal"].value_counts())

Répartition par statut :
statut
Honoré      1267
Planifié     231
Absent       206
Confirmé     160
Annulé       151
Name: count, dtype: int64

Top 5 spécialités :
specialite
Médecine générale          552
Pédiatrie                  306
Gynécologie-obstétrique    301
Cardiologie                186
Endocrinologie             166
Name: count, dtype: int64

Canaux de prise de RDV :
canal
Téléphone                 807
Guichet                   786
Recommandation médecin    402
Non renseigné              20
Name: count, dtype: int64


## 2. Effets réalistes attendus

Le dictionnaire de données annonce plusieurs effets réalistes :
- Pics de demande le **lundi** et le **samedi**, surtout le matin
- Hausse d'activité de **juillet à septembre** (saison des pluies)
- Taux d'absence d'environ **14 %**, plus élevé quand le délai est long et le patient non confirmé

In [3]:
jours_ordre = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
repartition = df.groupby("jour_semaine").size().reindex(jours_ordre)

fig = px.bar(
    x=repartition.index, y=repartition.values,
    labels={"x": "Jour", "y": "Nombre de RDV"},
    title="Répartition des RDV par jour de semaine",
    color=repartition.values, color_continuous_scale="Greens",
)
fig.update_layout(showlegend=False, height=400)
fig.show()

In [4]:
pivot = df.pivot_table(
    index="jour_semaine", columns="heure_int",
    values="id_rdv", aggfunc="count", fill_value=0
).reindex(jours_ordre)

fig = px.imshow(
    pivot, labels=dict(x="Heure", y="", color="Nb RDV"),
    title="Carte de chaleur : RDV par jour et par heure",
    color_continuous_scale="YlOrRd", aspect="auto", text_auto=True,
)
fig.update_layout(height=400)
fig.show()

In [5]:
volume = df.groupby("mois").size()
df_echus = df[df["statut"].isin(["Honoré", "Absent"])]
taux = df_echus.groupby("mois").apply(lambda x: (x["statut"] == "Absent").mean() * 100)

fig = go.Figure()
fig.add_trace(go.Bar(x=volume.index, y=volume.values, name="Volume", marker_color="#198754"))
fig.add_trace(go.Scatter(x=taux.index, y=taux.values, name="Taux d'absence (%)",
                          yaxis="y2", line=dict(color="#DC3545", width=3)))
fig.update_layout(
    title="Évolution mensuelle : volume et taux d'absence",
    yaxis=dict(title="Nombre de RDV"),
    yaxis2=dict(title="Taux d'absence (%)", overlaying="y", side="right"),
    hovermode="x unified", height=400,
)
fig.show()

## 3. Taux d'absence et facteurs associés

On calcule le taux d'absence global et on croise avec plusieurs variables pour identifier les facteurs de risque.

In [6]:
df_echus = df[df["statut"].isin(["Honoré", "Absent"])]
taux_global = (df_echus["statut"] == "Absent").mean() * 100
print(f"Taux d'absence global : {taux_global:.2f}%")
print(f"Objectif cahier des charges : 14% → 11%")

# Par spécialité
taux_spe = df_echus.groupby("specialite").apply(
    lambda x: (x["statut"] == "Absent").mean() * 100
).sort_values(ascending=False)
print("\nTaux d'absence par spécialité :")
print(taux_spe.round(2))

Taux d'absence global : 13.99%
Objectif cahier des charges : 14% → 11%

Taux d'absence par spécialité :
specialite
Cardiologie                18.38
Ophtalmologie              17.09
Kinésithérapie             16.18
ORL                        15.48
Endocrinologie             15.27
Dermatologie               15.09
Médecine générale          13.11
Pédiatrie                  13.06
Gynécologie-obstétrique     9.55
dtype: float64


In [7]:
df_echus = df_echus.copy()
df_echus["tranche_delai"] = pd.cut(
    df_echus["delai"],
    bins=[-30, 0, 7, 14, 30, 100],
    labels=["J-0", "1-7j", "8-14j", "15-30j", ">30j"],
)
taux_delai = df_echus.groupby("tranche_delai").apply(
    lambda x: (x["statut"] == "Absent").mean() * 100
)
print("Taux d'absence selon le délai de prise :")
print(taux_delai.round(2))

fig = px.bar(
    x=taux_delai.index.astype(str), y=taux_delai.values,
    labels={"x": "Délai RDV-prise", "y": "Taux d'absence (%)"},
    title="Taux d'absence selon le délai de prise",
    color=taux_delai.values, color_continuous_scale="Reds",
)
fig.update_layout(showlegend=False, height=400)
fig.show()

Taux d'absence selon le délai de prise :
tranche_delai
J-0       12.65
1-7j       9.99
8-14j     15.66
15-30j    20.28
>30j      38.30
dtype: float64


In [8]:
taux_conf = df_echus.groupby("confirme").apply(
    lambda x: (x["statut"] == "Absent").mean() * 100
)
print("Taux d'absence selon la confirmation :")
print(taux_conf.round(2))

Taux d'absence selon la confirmation :
confirme
Non    20.27
Oui     8.87
dtype: float64


## 4. Prévision de la demande (EF-24)

Nous comparons deux modèles :
- **Baseline naïve** : moyenne des 4 dernières semaines
- **Holt-Winters** : lissage exponentiel avec tendance et saisonnalité

Évaluation sur les 8 dernières semaines tenues à l'écart (MAE / MAPE).

In [9]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error

df["semaine"] = df["date_rdv"].dt.to_period("W").apply(lambda p: p.start_time)
serie = df.groupby(["semaine", "specialite"]).size().unstack(fill_value=0)
toutes_semaines = pd.date_range(serie.index.min(), serie.index.max(), freq="W-MON")
serie = serie.reindex(toutes_semaines, fill_value=0)

NB_TEST = 8
train, test = serie.iloc[:-NB_TEST], serie.iloc[-NB_TEST:]

resultats = []
for spe in serie.columns:
    y_train, y_test = train[spe], test[spe]
    pred_naive = pd.Series([y_train.iloc[-4:].mean()] * NB_TEST, index=test.index)
    mae_n = mean_absolute_error(y_test, pred_naive)
    mape_n = np.mean(np.abs((y_test - pred_naive) / y_test.replace(0, 1))) * 100

    try:
        model = ExponentialSmoothing(y_train, trend="add", seasonal="add",
                                     seasonal_periods=13, initialization_method="estimated")
        fit = model.fit(optimized=True)
        pred_hw = fit.forecast(NB_TEST)
        mae_hw = mean_absolute_error(y_test, pred_hw)
        mape_hw = np.mean(np.abs((y_test - pred_hw) / y_test.replace(0, 1))) * 100
    except Exception:
        mae_hw, mape_hw = np.nan, np.nan

    resultats.append({
        "specialite": spe,
        "MAPE_naive": round(mape_n, 1),
        "MAPE_HW": round(mape_hw, 1) if not np.isnan(mape_hw) else None,
        "MAE_naive": round(mae_n, 1),
        "MAE_HW": round(mae_hw, 1) if not np.isnan(mae_hw) else None,
    })

pd.DataFrame(resultats).sort_values("MAPE_naive")

,specialite,MAPE_naive,MAPE_HW,MAE_naive,MAE_HW
5,Médecine générale,24.4,32.3,1.8,2.1
8,Pédiatrie,34.8,30.0,1.8,1.6
3,Gynécologie-obstétrique,46.9,69.8,1.5,2.3
2,Endocrinologie,57.0,152.3,0.9,1.9
4,Kinésithérapie,59.9,61.5,0.8,1.1
7,Ophtalmologie,60.0,89.2,1.0,1.4
6,ORL,62.0,61.0,1.2,0.9
1,Dermatologie,63.0,87.6,1.2,1.3
0,Cardiologie,111.2,181.4,1.6,2.3


**Observation** : avec seulement 66 semaines d'historique, la baseline naïve est souvent meilleure que Holt-Winters (MAPE 25-110 %). Identifier une saisonnalité annuelle robuste nécessite 2-3 ans de données. Le modèle reste utile pour les spécialités régulières (Pédiatrie, Gynécologie). Les prévisions finales sont sauvegardées dans `data/propre/previsions.json`.

## 5. Score de risque d'absence (EF-25)

Deux modèles de classification :
- **Régression logistique** (interprétable)
- **Random Forest** (performant)

Évaluation : AUC, précision, rappel, matrice de confusion au seuil optimal coût/bénéfice.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix

df_model = df[df["statut"].isin(["Honoré", "Absent"])].copy()
df_model["est_confirme"] = (df_model["confirme"] == "Oui").astype(int)
df_model["cible"] = (df_model["statut"] == "Absent").astype(int)

hist = df_model.groupby("patient_id")["cible"].agg(["sum", "count"])
hist["taux_hist"] = hist["sum"] / hist["count"]
df_model = df_model.merge(hist[["taux_hist"]], left_on="patient_id", right_index=True, how="left")
df_model["taux_hist"] = df_model["taux_hist"].fillna(0)

df_model = pd.get_dummies(df_model, columns=["specialite"], prefix="spe")
features = ["delai", "heure_int", "age", "est_confirme", "taux_hist"] + \
           [c for c in df_model.columns if c.startswith("spe_")]

X = df_model[features].fillna(0)
y = df_model["cible"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Modèle 1 : Régression logistique
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train_s, y_train)
proba_lr = lr.predict_proba(X_test_s)[:, 1]

# Modèle 2 : Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]

# Métriques
print(f"AUC — Régression logistique : {roc_auc_score(y_test, proba_lr):.3f}")
print(f"AUC — Random Forest        : {roc_auc_score(y_test, proba_rf):.3f}")

importances = pd.DataFrame({
    "variable": X.columns,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False).head(8)
print("\nTop variables (Random Forest) :")
print(importances.to_string(index=False))

AUC — Régression logistique : 0.960
AUC — Random Forest        : 0.955

Top variables (Random Forest) :
                   variable  importance
                  taux_hist    0.679345
                      delai    0.084703
                        age    0.081495
                  heure_int    0.051433
               est_confirme    0.048150
              spe_Pédiatrie    0.007855
      spe_Médecine générale    0.007768
spe_Gynécologie-obstétrique    0.006571


In [ ]:
fig = px.bar(
    importances, x="importance", y="variable", orientation="h",
    title="Importance des variables — Random Forest",
    color="importance", color_continuous_scale="Greens",
)
fig.update_layout(showlegend=False, height=400,
                  yaxis=dict(categoryorder="total ascending"))
fig.show()

: 

## 6. Limites et perspectives

### Limites
- **Données synthétiques** : les résultats illustrent la méthode, pas la réalité de la clinique.
- **Historique court** (66 semaines) : insuffisant pour identifier une saisonnalité annuelle robuste.
- **Biais possibles** : la distribution d'absence peut varier selon les effets réalistes imposés (pics, saisonnalité).
- **Variables manquantes** : pas de météo, pas de rappel effectif, pas de dossier médical.
- **Dérive temporelle** : un modèle entraîné sur 2025-2026 devra être réentraîné régulièrement.

### Perspectives
- Réentraînement trimestriel des modèles (EF-24 / EF-25).
- Ajout de variables externes (saison des pluies, jours fériés, motifs de consultation).
- Module de rappels SMS réel pour valider le score d'absence en production.
- Généralisation aux autres cliniques du groupe.

### Conclusion
Le socle technique et analytique est opérationnel. Les indicateurs du tableau de bord permettent un pilotage quotidien. Les deux modèles (prévision et score d'absence) fournissent une aide à la décision, sans jamais bloquer la prise de rendez-vous ni exposer le patient.